# 01 · Data quality & exploratory analysis

**AeroCast-AI · Project 21 · Phase 1.** Dataset: `nycflights13`: every 2013 departure from EWR, JFK and LGA (336,776 flights, 105 destinations, 16 carriers).

| Stage | Notebook |
|---|---|
| **1. Data quality & EDA** | **this notebook** |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| 4. Evaluation | `04_evaluation.ipynb` |

What this notebook establishes:
1. The data is **internally consistent**, and its missing values have a clear meaning (cancellations and diversions).
2. The **decision time** (end of day D-1), and which columns are therefore allowed as features.
3. The patterns in the data that motivate each feature family and the evaluation design.

All logic lives in `src/aerocast/` (`data`, `eda`, `plots`). This notebook only calls it and explains the output.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from aerocast import data, eda, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)

## 1. Load

`data.load_raw()` parses the date, builds a scheduled-departure timestamp, and sorts chronologically. `data.add_target()` adds `is_cancelled`, `is_diverted` and the target `is_delayed`.

In [2]:
raw = data.load_raw()
df = data.add_target(raw)
print(raw.shape)
raw.head()

(336776, 23)


,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,origin,dest,air_time,distance,hour,minute,time_hour,name,date,sched_dep_dt
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:15:00
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:29:00
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.,2013-01-01,2013-01-01 05:40:00
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways,2013-01-01,2013-01-01 05:45:00
4,5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,...,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:58:00


In [3]:
raw.dtypes.to_frame("dtype").T

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,origin,dest,air_time,distance,hour,minute,time_hour,name,date,sched_dep_dt
dtype,int64,int64,int64,int64,float64,int64,float64,float64,int64,float64,...,str,str,float64,int64,int64,int64,str,str,datetime64[us],datetime64[us]


## 2. Data quality audit

### 2.1 Missingness

In [4]:
eda.missingness(raw)

,n_missing,pct
arr_delay,9430,2.8001
air_time,9430,2.8001
arr_time,8713,2.5872
dep_time,8255,2.4512
dep_delay,8255,2.4512
tailnum,2512,0.7459


The missing values are **structural, not random**:
* `dep_time` / `dep_delay` missing (8,255 rows) means the flight was **cancelled**: it never departed.
* `arr_delay` / `air_time` missing while `dep_time` is present means it was **diverted** (or the arrival wasn't recorded).
* `tailnum` missing (2,512 rows) means no aircraft was assigned. Every one of these is also a cancellation (see below).

So imputing these columns would be wrong. They encode outcomes, and outcomes are what we predict.

### 2.2 Integrity: duplicates, cancellations, diversions

In [5]:
eda.quality_audit(df).to_frame()

,value
rows,336776
duplicate id,0
"duplicate (date, carrier, flight, origin, sched_dep)",0
cancelled (no dep_time),8255
"diverted (departed, no arr_delay)",1175
missing tailnum,2512
missing tailnum AND cancelled,2512


### 2.3 Internal consistency between redundant columns

In [6]:
eda.consistency_checks(df).to_frame()

,value
hour/minute match sched_dep_time (share),1.0
"dep_delay = dep_time - sched_dep_time, mod 24h (share)",1.0
routes with >1 distinct distance,2.0
air_time <= 0,0.0


* No duplicate records, either by id or by natural key (date, carrier, flight number, origin, scheduled time).
* `hour`/`minute` always match `sched_dep_time`, and `dep_delay` equals `dep_time − sched_dep_time` (mod 24 h) for **100%** of departed flights, so the delay field is trustworthy.
* Only 2 routes have more than one distance value, which is negligible. No zero or negative air times.

**Conclusion:** no cleaning is needed beyond handling cancelled and diverted flights.

## 3. Decision time and leakage audit

The brief requires that **no post-departure information** is used for a pre-departure prediction. We fix the decision time first:

> **Predict at the end of day D-1 (day-ahead).** Allowed inputs: the published timetable for day D, plus realised outcomes of flights on dates **strictly before D**.

Every raw column is classified below. Only the first three rows may describe the flight being predicted.

In [7]:
eda.leakage_audit()

,columns,source,known at D-1?,use
0,"year, month, day, hour, minute, sched_dep_time, sched_arr_time",schedule,yes,calendar & time-of-day features
1,"carrier, flight, origin, dest, distance",schedule,yes,identity / route features
2,tailnum,aircraft assignment,yes (assumed),rotation features; assumes routing is published day-ahead
3,"dep_time, dep_delay",operations,NO - after departure,never a feature; aggregated over past dates only
4,"arr_time, arr_delay, air_time",operations,NO - after arrival,target + aggregated over past dates only
5,"id, name, time_hour",bookkeeping,-,dropped (row id / redundant)


Two consequences that are easy to get wrong:
* `arr_delay` / `dep_delay` **are** used, but only aggregated over **past dates** (e.g. "this route's delay rate over the last 28 days").
* **Cancelled flights stay in the schedule counts.** At D-1 nobody knows a flight will be cancelled, so it still counts toward airport congestion. It's only dropped from the *target*.

## 4. Target

`is_delayed = arr_delay ≥ 15 min`, the BTS definition of a late arrival. Cancelled and diverted flights (2.8%) have no arrival delay and are excluded from modelling.

In [8]:
fig = plots.delay_histogram(df["arr_delay"]); plots.save(fig, "eda_delay_hist"); fig

<Figure size 700x300 with 1 Axes>

In [9]:
eda.delay_shares(df).to_frame()

,share of completed flights
>= 15 min,0.2447
>= 30 min,0.1613
>= 60 min,0.0865


The distribution is strongly right-skewed with a long tail. At the 15-minute threshold, **24.5%** of completed flights are late. That's moderate class imbalance, so **PR-AUC** is the main ranking metric: it measures performance on the positive (late) class, which accuracy and ROC-AUC can hide. We don't re-weight the classes, because we want the predicted probabilities to stay calibrated.

## 5. When are flights late?

In [10]:
fig = plots.rate_by_time(eda.rate_by(df, "month")["rate"],
                         eda.rate_by(df, "hour")["rate"].dropna(),
                         eda.rate_by(df, df["date"].dt.dayofweek)["rate"])
plots.save(fig, "eda_rate_by_time"); fig

<Figure size 1300x330 with 3 Axes>

In [11]:
eda.rate_by(df, "month").T

month,1,2,3,4,5,6,7,8,9,10,11,12
rate,0.2349,0.2378,0.236,0.2875,0.2267,0.3152,0.3221,0.246,0.1372,0.1756,0.181,0.3355
n,26398.0000,23611.0000,27902.000,27564.0000,28128.0000,27075.0000,28293.0000,28756.000,27010.0000,28618.0000,26971.000,27020.0000


* **Time of day is the dominant pattern.** The delay rate rises from 10% at 05:00 to 37% at 21:00. Delay *propagates* through the day as aircraft, crews and gates back up. → cyclic departure/arrival time features.
* **Strong seasonality, but only one year.** June–July and December peak, while September–October are the calmest months. Each month appears only once, so a month feature can't generalise to unseen months. → no month feature (see notebook 02).
* **The chronological split is a built-in stress test.** The calmest months (Sep–Oct) become the calibration window and the worst month (Dec) is in test. This matters a lot for calibration (notebook 04).

In [12]:
fig = plots.rate_by_group(eda.rate_by(df, "carrier"), eda.rate_by(df, "origin"))
plots.save(fig, "eda_rate_by_carrier_origin"); fig

<Figure size 1100x360 with 2 Axes>

Carrier differences are large (about 3× between best and worst), but several carriers are tiny (OO has 32 flights). Two consequences: the one-hot encoder groups categories with fewer than 50 flights (`min_frequency=50`), and the historical rates are **shrunk toward a prior** so that small groups don't produce extreme estimates (notebook 02). The three origins differ less than the carriers.

## 6. Is yesterday informative?

This tests whether a *past-only* feature can carry signal at all.

In [13]:
daily = eda.daily_rate(df)
acf = eda.autocorrelation(daily)
fig = plots.daily_rate_and_acf(daily, acf); plots.save(fig, "eda_daily_autocorr"); fig

<Figure size 1200x320 with 2 Axes>

In [14]:
acf.round(3).to_frame().T

,1,2,3,4,5,6,7,8,9,10,11,12,13,14
autocorr,0.398,0.128,0.161,0.152,0.107,0.139,0.112,0.083,0.089,0.103,0.18,0.148,0.145,0.195


Lag-1 autocorrelation is about **0.40**, then it drops quickly to about 0.1–0.15. Disruption is **bursty** (storms, air-traffic-control programmes) and has short memory. This motivates two kinds of history feature:
* **day-level** (yesterday's network-wide and per-origin delay rate): captures yesterday's shock;
* **entity-level** (route, carrier, aircraft, flight-number rates over 7/28 days): captures stable propensities.

Notebook 02 explains why the day-level group needs separate treatment.

## 7. Scheduled congestion vs delay

In [15]:
curve = eda.congestion_curve(df)
fig = plots.congestion_curve(curve); plots.save(fig, "eda_congestion"); fig

<Figure size 700x300 with 1 Axes>

Busier departure banks have more delays. But busy banks are concentrated in the evening, when delays are already high, so this is partly **confounded with time of day**. Congestion features are still included (they're legitimately known from the schedule), and the ablation in notebook 04 tests whether they add anything *beyond* time of day.

## 8. EDA → design decisions

In [16]:
eda.design_decisions()

,EDA observation,design decision
0,Missingness = cancellations / diversions,"drop from target, keep in schedule (congestion) counts"
1,Strong time-of-day gradient (10% -> 37%),cyclic dep/arr time + raw hour
2,"One year; seasonal months; Sep-Oct calm, Dec extreme",no month feature; chronological split; calibration studied under shift
3,Short-memory daily shocks (lag-1 autocorr ~0.4),1-day / 7-day lagged rates (day-level group)
4,Stable carrier / route / aircraft propensities; some tiny groups,28-day entity rates with Bayesian shrinkage; one-hot min_frequency=50
5,Congestion correlates with delay but is confounded with hour,"scheduled-bank counts, kept and tested by ablation"
6,Leakage risk from post-departure columns,D-1 decision time; past-only history; automated leakage test
